In [8]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.ensemble import RandomForestRegressor

CSV_PATH = "police.csv"       
OUTPUT_JSON = "hotspot_predictions.json"

pd.set_option("display.max_colwidth", 60)

In [9]:
print("Loading", CSV_PATH, "...")
df = pd.read_csv(
    CSV_PATH,
    usecols=["id","latitude","longitude","location","vehicle_type","violation_type",
             "created_datetime","junction_name","validation_status","validation_timestamp"]
)
print(f"Loaded {len(df):,} rows")


df["created_datetime"] = pd.to_datetime(df["created_datetime"], format="mixed", utc=True)
df["hour"] = df["created_datetime"].dt.hour
df["day_of_week"] = df["created_datetime"].dt.weekday  


df["near_junction"] = (df["junction_name"] != "No Junction").astype(int)
road_from_location = df["location"].str.split(",").str[0].str.strip()
df["junction_clean"] = np.where(df["near_junction"] == 1, df["junction_name"], road_from_location)

zone_counts = df.groupby("junction_clean").agg(
    violation_count=("id", "count"),
    lat=("latitude", "mean"),
    lon=("longitude", "mean"),
    near_junction=("near_junction", "max"),
).reset_index().sort_values("violation_count", ascending=False)

top20 = zone_counts.head(20).reset_index(drop=True)
top20["rank"] = top20.index + 1

print("\nTop 20 zones by violation count:")
print(top20[["rank","junction_clean","violation_count","near_junction"]].to_string(index=False))

df_top = df[df["junction_clean"].isin(top20["junction_clean"])].copy()
print(f"\nRecords in top 20 zones: {len(df_top):,}")

Loading police.csv ...
Loaded 298,450 rows

Top 20 zones by violation count:
 rank                         junction_clean  violation_count  near_junction
    1         BTP051 - Safina Plaza Junction            15449              1
    2            BTP082 - KR Market Junction            11538              1
    3                BTP040 - Elite Junction            10718              1
    4        BTP044 - Sagar Theatre Junction            10549              1
    5                        Outer Ring Road             8817              0
    6                               MBT Road             6428              0
    7               New Horizon College Road             6291              0
    8       BTP211 - Central Street Junction             5388              1
    9             BTP058 - Subbanna Junction             5189              1
   10                           Unnamed Road             5061              0
   11                     Sahakar Nagar Road             4715              0

Loaded 298,450 rows



Top 20 zones by violation count:
 rank                         junction_clean  violation_count  near_junction
    1         BTP051 - Safina Plaza Junction            15449              1
    2            BTP082 - KR Market Junction            11538              1
    3                BTP040 - Elite Junction            10718              1
    4        BTP044 - Sagar Theatre Junction            10549              1
    5                        Outer Ring Road             8817              0
    6                               MBT Road             6428              0
    7               New Horizon College Road             6291              0
    8       BTP211 - Central Street Junction             5388              1
    9             BTP058 - Subbanna Junction             5189              1
   10                           Unnamed Road             5061              0
   11                     Sahakar Nagar Road             4715              0
   12          BTP027 - Modi Bridge Juncti

In [10]:
grp = df_top.groupby(["junction_clean", "day_of_week", "hour"]).size().reset_index(name="violation_count")

all_combos = pd.MultiIndex.from_product(
    [top20["junction_clean"], range(7), range(24)],
    names=["junction_clean", "day_of_week", "hour"]
).to_frame(index=False)
full = all_combos.merge(grp, on=["junction_clean", "day_of_week", "hour"], how="left")
full["violation_count"] = full["violation_count"].fillna(0).astype(int)
full = full.merge(top20[["junction_clean", "lat", "lon", "near_junction", "rank"]], on="junction_clean")

print(f"Training table: {len(full)} rows (20 zones x 7 days x 24 hours = {20*7*24})")
print(f"Violation count stats: min={full['violation_count'].min()}, "
      f"max={full['violation_count'].max()}, mean={full['violation_count'].mean():.2f}")

zone_list = top20.sort_values("rank")["junction_clean"].tolist()
zone_to_id = {z: i for i, z in enumerate(zone_list)}
full["zone_id"] = full["junction_clean"].map(zone_to_id)
full["hour_sin"] = np.sin(2 * np.pi * full["hour"] / 24)
full["hour_cos"] = np.cos(2 * np.pi * full["hour"] / 24)
full["dow_sin"] = np.sin(2 * np.pi * full["day_of_week"] / 7)
full["dow_cos"] = np.cos(2 * np.pi * full["day_of_week"] / 7)

FEATURES = ["zone_id", "day_of_week", "hour", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "near_junction"]
X = full[FEATURES].values
y = full["violation_count"].values

print(f"\nTraining RandomForestRegressor on {len(X)} samples, {len(FEATURES)} features...")
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X, y)

mae = np.mean(np.abs(model.predict(X) - y))
print(f"In-sample MAE: {mae:.2f} violations (mean count: {y.mean():.2f})")

Training table: 3360 rows (20 zones x 7 days x 24 hours = 3360)
Violation count stats: min=0, max=835, mean=36.83

Training RandomForestRegressor on 3360 samples, 8 features...
In-sample MAE: 8.91 violations (mean count: 36.83)


In-sample MAE: 8.81 violations (mean count: 36.83)


In [11]:
df_top["validation_timestamp"] = pd.to_datetime(df_top["validation_timestamp"], format="mixed", utc=True, errors="coerce")
approved = df_top[df_top["validation_status"] == "approved"].copy()
approved["response_hrs"] = (approved["validation_timestamp"] - approved["created_datetime"]).dt.total_seconds() / 3600
approved["response_hrs"] = approved["response_hrs"].where(approved["response_hrs"].between(0, 200))

resp_stats = approved.groupby("junction_clean")["response_hrs"].agg(["mean", "median"]).reset_index()
resp_stats.columns = ["junction_clean", "avg_response_hrs", "median_response_hrs"]

zones = top20.merge(resp_stats, on="junction_clean", how="left")
city_median_resp = approved["response_hrs"].median()
zones["avg_response_hrs"] = zones["avg_response_hrs"].fillna(city_median_resp)
zones["median_response_hrs"] = zones["median_response_hrs"].fillna(city_median_resp)

resp_thresh = zones["avg_response_hrs"].median()
vol_thresh = zones["violation_count"].median()
zones["under_resourced"] = (zones["avg_response_hrs"] > resp_thresh) & (zones["violation_count"] > vol_thresh)

print(zones[["rank","junction_clean","violation_count","avg_response_hrs","under_resourced"]].round(1).to_string(index=False))

 rank                         junction_clean  violation_count  avg_response_hrs  under_resourced
    1         BTP051 - Safina Plaza Junction            15449              41.7             True
    2            BTP082 - KR Market Junction            11538              39.7            False
    3                BTP040 - Elite Junction            10718              37.2            False
    4        BTP044 - Sagar Theatre Junction            10549              41.9             True
    5                        Outer Ring Road             8817              42.6             True
    6                               MBT Road             6428              39.4            False
    7               New Horizon College Road             6291              41.8             True
    8       BTP211 - Central Street Junction             5388              40.5            False
    9             BTP058 - Subbanna Junction             5189              35.2            False
   10                         

In [12]:
rows = []
for z in zone_list:
    zid = zone_to_id[z]
    zone_row = zones[zones["junction_clean"] == z].iloc[0]
    for d in range(7):
        for h in range(24):
            hour_sin = np.sin(2*np.pi*h/24); hour_cos = np.cos(2*np.pi*h/24)
            dow_sin = np.sin(2*np.pi*d/7); dow_cos = np.cos(2*np.pi*d/7)
            feat = [[zid, d, h, hour_sin, hour_cos, dow_sin, dow_cos, int(zone_row["near_junction"])]]
            tree_preds = np.array([t.predict(feat)[0] for t in model.estimators_])
            rows.append({
                "zone": z, "day": d, "hour": h,
                "predicted": tree_preds.mean(), "std": tree_preds.std()
            })

pred_df = pd.DataFrame(rows)

pred_df["zone_max"] = pred_df.groupby("zone")["predicted"].transform("max").clip(lower=1)
pred_df["intensity_ratio"] = pred_df["predicted"] / pred_df["zone_max"]
pred_df["label"] = np.select(
    [pred_df["intensity_ratio"] > 0.65, pred_df["intensity_ratio"] > 0.35],
    ["High", "Medium"], default="Low"
)

pred_df["cv"] = pred_df["std"] / pred_df["predicted"].clip(lower=1)
cv_min, cv_max = pred_df["cv"].quantile(0.05), pred_df["cv"].quantile(0.95)
pred_df["cv_norm"] = ((pred_df["cv"] - cv_min) / (cv_max - cv_min)).clip(0, 1)
pred_df["confidence"] = (96 - pred_df["cv_norm"] * 28).round().astype(int).clip(65, 97)

print("Label distribution:")
print(pred_df["label"].value_counts())
print()
print(pred_df[["predicted","intensity_ratio","confidence"]].describe().round(2))

Label distribution:
label
Low       2349
Medium     679
High       332
Name: count, dtype: int64

       predicted  intensity_ratio  confidence
count    3360.00          3360.00     3360.00
mean       36.95             0.24       88.04
std        55.44             0.27        7.26
min         0.00             0.00       68.00
25%         0.32             0.00       86.00
50%        18.06             0.14       90.00
75%        53.00             0.41       93.00
max       665.83             1.00       96.00


In [13]:
zone_idx = {z: i for i, z in enumerate(zone_list)}
LABEL_CODE = {"Low": 0, "Medium": 1, "High": 2}

grid = [[[None]*24 for _ in range(7)] for _ in range(20)]
for _, row in pred_df.iterrows():
    zi = zone_idx[row["zone"]]
    d = int(row["day"]); h = int(row["hour"])
    grid[zi][d][h] = [round(float(row["predicted"])), LABEL_CODE[row["label"]], int(row["confidence"])]

zones_sorted = zones.sort_values("rank").reset_index(drop=True)
zones_out = []
for _, z in zones_sorted.iterrows():
    zones_out.append({
        "name": z["junction_clean"],
        "lat": round(float(z["lat"]), 4),
        "lon": round(float(z["lon"]), 4),
        "nearJunc": bool(z["near_junction"]),
        "totalViolations": int(z["violation_count"]),
        "avgResp": round(float(z["avg_response_hrs"]), 1),
        "underResourced": bool(z["under_resourced"]),
        "rank": int(z["rank"])
    })

output = {
    "meta": {
        "model": "RandomForestRegressor (300 trees), trained on real Bengaluru parking violation records",
        "trainedRows": len(df_top),
        "labelCodes": {"0": "Low", "1": "Medium", "2": "High"},
        "dayOrder": ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"],
        "generatedRows": 3360
    },
    "zones": zones_out,
    "grid": grid
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output, f, separators=(",", ":"))

size_kb = os.path.getsize(OUTPUT_JSON) / 1024
print(f"Saved {OUTPUT_JSON} ({size_kb:.1f} KB)")
print(f"Zones: {len(zones_out)}  |  Grid cells: 20 x 7 x 24 = {20*7*24}")

Saved hotspot_predictions.json (35.2 KB)
Zones: 20  |  Grid cells: 20 x 7 x 24 = 3360


In [15]:
import matplotlib.pyplot as plt

# One point = one zone/day/hour prediction
plt.figure(figsize=(10, 6))

plt.scatter(
    pred_df["predicted"],
    pred_df["confidence"],
    alpha=0.6,
    s=40
)

plt.xlabel("Predicted Violations")
plt.ylabel("Confidence (%)")
plt.title("Random Forest Ensemble Confidence")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("rf_confidence_scatter.png", dpi=300)
plt.close()

print("Saved rf_confidence_scatter.png")

Saved rf_confidence_scatter.png
